# Módulo 04: Optimización Avanzada, AQE y Estrategias de Particionado

## 1. Adaptive Query Execution (AQE): La Revolución en Tiempo de Ejecución

En las versiones clásicas de Apache Spark (pre-3.0), el optimizador Catalyst generaba un plan físico estático basado en estimaciones previas. Si las estadísticas estaban desactualizadas o un filtro reducía drásticamente los datos, Spark continuaba ejecutando un plan ineficiente (por ejemplo, manteniendo 200 particiones de shuffle para apenas unos cuantos kilobytes de datos).

A partir de Spark 3.x, **AQE** reoptimiza las consultas dinámicamente entre etapas (*Stages*) utilizando métricas recolectadas en tiempo real:

1. **Coalesce dinámico de particiones de shuffle (`coalescePartitions`):** Fusiona automáticamente particiones de shuffle contiguas y pequeñas para evitar la sobrecarga de programar miles de tareas diminutas.
2. **Conversión dinámica a Broadcast Join:** Si tras aplicar un filtro el volumen de una tabla desciende por debajo de `spark.sql.autoBroadcastJoinThreshold`, Spark conmuta dinámicamente de Sort-Merge Join a Broadcast Hash Join en caliente.
3. **Manejo dinámico de sesgo de datos (`skewJoin`):** Detecta particiones excesivamente pesadas que ralentizan la etapa (*stragglers*) y las subdivide automáticamente en subtareas paralelas.

---

## 2. Diferencias Críticas: `repartition()` vs. `coalesce()`

| Característica | `repartition(n)` | `coalesce(n)` |
| :--- | :--- | :--- |
| **Mecanismo** | *Full Shuffle* a través de la red | Fusión de particiones locales contiguas |
| **Costo Computacional** | Alto (I/O de red, serialización y disco) | Mínimo (sin movimiento de datos entre nodos) |
| **Capacidad de escala** | Puede **aumentar** o **disminuir** particiones | **Solo puede disminuir** particiones |
| **Distribución de datos** | Uniforme y balanceada (Round-Robin o Hash) | Puede generar particiones de tamaños desiguales |
| **Caso de uso principal** | Balancear datos antes de operaciones pesadas | Reducir el número de archivos antes de guardar a disco |

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Inicialización canónica habilitando explícitamente las tres optimizaciones de AQE
spark = (
    SparkSession.builder
    .appName("04_Optimizacion_AQE_Particionado")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.sql.shuffle.partitions", "16")  # Configuración base para pruebas
    .getOrCreate()
)

print(f"AQE habilitado: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Coalesce dinámico: {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")
print(f"Manejo de Skew Join: {spark.conf.get('spark.sql.adaptive.skewJoin.enabled')}")

AQE habilitado: true
Coalesce dinámico: true
Manejo de Skew Join: true


## 3. Demostración Física de Particionado: Repartition frente a Coalesce

Para inspeccionar el número de particiones físicas subyacentes en un DataFrame, accedemos temporalmente a la interfaz de su RDD mediante `.rdd.getNumPartitions()`.

A continuación, creamos un rango sintético y comparamos el comportamiento de ambas transformaciones.

In [2]:
# 1. Crear un dataset sintético en memoria
df_base = spark.range(0, 500000)
particiones_iniciales = df_base.rdd.getNumPartitions()
print(f"1. Particiones físicas iniciales: {particiones_iniciales}")

# 2. repartition(8): Obliga a redistribuir mediante un Full Shuffle
df_repartition = df_base.repartition(8)
print(f"2. Particiones tras repartition(8): {df_repartition.rdd.getNumPartitions()}")

# 3. coalesce(2): Reduce particiones fusionándolas sin Full Shuffle
df_coalesce = df_repartition.coalesce(2)
print(f"3. Particiones tras coalesce(2): {df_coalesce.rdd.getNumPartitions()}")

# 4. Intento de incrementar particiones con coalesce (no surte efecto hacia arriba)
df_intento_subida = df_coalesce.coalesce(10)
print(f"4. Particiones tras coalesce(10) intentando subir: {df_intento_subida.rdd.getNumPartitions()}")

1. Particiones físicas iniciales: 8
2. Particiones tras repartition(8): 8
3. Particiones tras coalesce(2): 2
4. Particiones tras coalesce(10) intentando subir: 8


## 4. Inspección del Plan Físico bajo AQE

Cuando AQE está habilitado, Spark compila el plan de ejecución encapsulando los operadores en nodos `AdaptiveSparkPlan`. Si observamos el plan antes de materializar una acción, veremos los operadores iniciales; al ejecutarse la consulta, AQE inyecta estadísticas en tiempo real y ajusta la topología final.

In [3]:
# Consulta con agregación que genera una etapa de Shuffle
df_agrupado = (
    df_repartition
    .withColumn("grupo", F.col("id") % 4)
    .groupBy("grupo")
    .agg(
        F.count("id").alias("total_registros"),
        F.sum("id").alias("suma_total")
    )
)

# Visualizar el plan formateado con nodos adaptativos
df_agrupado.explain(mode="formatted")

# Ejecución de la acción
df_agrupado.show()

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Exchange (2)
               +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 500000, step=1, splits=Some(8))

(2) Exchange
Input [1]: [id#0L]
Arguments: RoundRobinPartitioning(8), REPARTITION_BY_NUM, [plan_id=74]

(3) Project
Output [2]: [id#0L, (id#0L % 4) AS grupo#8L]
Input [1]: [id#0L]

(4) HashAggregate
Input [2]: [id#0L, grupo#8L]
Keys [1]: [grupo#8L]
Functions [2]: [partial_count(1), partial_sum(id#0L)]
Aggregate Attributes [2]: [count#20L, sum#21L]
Results [3]: [grupo#8L, count#22L, sum#23L]

(5) Exchange
Input [3]: [grupo#8L, count#22L, sum#23L]
Arguments: hashpartitioning(grupo#8L, 16), ENSURE_REQUIREMENTS, [plan_id=79]

(6) HashAggregate
Input [3]: [grupo#8L, count#22L, sum#23L]
Keys [1]: [grupo#8L]
Functions [2]: [count(1), sum(id#0L)]
Aggregate Attributes [2]: [count(1)#13L, sum(id#0L)#15L]
Results [3]: [grupo#8

## 5. Reto Práctico: Optimización de Salida y Reducción Controlada

### Instrucciones del Ejercicio:
1. Toma el DataFrame base `df_base`.
2. Filtra los registros para conservar únicamente aquellos cuyo campo `"id"` sea múltiplo de 50.
3. Reduce las particiones del resultado a **exactamente 1 partición** utilizando la estrategia más óptima (sin incurrir en un *Full Shuffle*).
4. Calcula el promedio del campo `"id"` (`F.avg("id")`) y extrae el valor escalar resultante.
5. Verifica con aserciones formales tanto la cantidad de particiones finales como el valor del promedio calculado.

In [4]:
# 1. Filtrado de múltiplos de 50
df_filtrado = df_base.filter(F.col("id") % 50 == 0)

# 2. Reducción eficiente de particiones previa a consolidación
df_optimizado = df_filtrado.coalesce(1)

# 3. Validación de particiones físicas
particiones_resultado = df_optimizado.rdd.getNumPartitions()
print(f"Particiones resultantes: {particiones_resultado}")

# 4. Cálculo del promedio escalar
promedio_calculado = (
    df_optimizado
    .select(F.avg("id").alias("promedio"))
    .collect()[0]["promedio"]
)
print(f"Promedio calculado: {promedio_calculado}")

# 5. Aserciones formales de control
# Rango 0 a 499950 en pasos de 50: promedio = (0 + 499950) / 2 = 249975.0
assert particiones_resultado == 1, f"Error: Se esperaba 1 partición, se obtuvieron {particiones_resultado}"
assert promedio_calculado == 249975.0, f"Error: Promedio esperado 249975.0, obtenido {promedio_calculado}"

print("¡Aserción aprobada! Cuaderno 04 completado exitosamente.")

Particiones resultantes: 1
Promedio calculado: 249975.0
¡Aserción aprobada! Cuaderno 04 completado exitosamente.
